# Træn robust YOLOv11s — FCM Landmark Detection (v3)

Træner med **dobbelt augmentation**:
1. **Data-niveau:** dust-pletter, ridser og lys-gradient bages ind i de syntetiske billeder
2. **Trænings-niveau:** mixup, copy-paste, perspektiv, shear oven i flips/rotation/mosaic

Output: `models/v3_augmented/best.pt` (efter download).

## Sådan kører du

1. **Runtime → Change runtime type → T4 GPU**
2. **Runtime → Run all** (Cmd/Ctrl + F9)
3. Vent ~10-15 min (lidt længere pga. flere augmentations)
4. Sidste celle downloader `best.pt`
5. På din Mac:
   ```bash
   mkdir -p models/v3_augmented
   mv ~/Downloads/best.pt models/v3_augmented/
   ./switch_model.sh v3_augmented
   ```

## 1. Setup

In [ ]:
!pip install ultralytics==8.4.47 -q

import ultralytics, torch
print(f'Ultralytics: {ultralytics.__version__}')
print(f'PyTorch:     {torch.__version__}')
print(f'CUDA:        {torch.cuda.is_available()}')
print(f'GPU:         {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "INGEN — skift runtime til T4!"}')

## 2. Datagenerator + plade-imperfektioner

Hvert billede får tilfældig kombination af:
- Støj-niveau (kalibreret/realistisk/svært)
- Frø-størrelse (mindre/normal/større)
- Baggrunds-reflektans (0.75-0.95)
- **NYT i v3:** dust-pletter (50% sandsynlighed), ridser (30%), lys-gradient (70%)

Pointen: modellen ser "plade-imperfektioner" der minder om rigtige Videometer-data.

In [ ]:
import numpy as np
from pathlib import Path
import shutil
from PIL import Image

# Presets — uændret fra v2
NOISE_LEVELS = [0.02, 0.05, 0.10]
SEED_SIZE_PRESETS = [
    (8, 12, 3, 5),    # mindre
    (12, 18, 5, 7),   # normal
    (18, 24, 7, 9),   # større
]
BACKGROUND_REFLECTANCE_MIN = 0.75
BACKGROUND_REFLECTANCE_MAX = 0.95
SEED_REFLECTANCE_MIN, SEED_REFLECTANCE_MAX = 0.20, 0.45
PACKING_GAP_PIXELS = 2
MAX_PLACEMENT_ATTEMPTS = 200

# v3-tilføjelser
DUST_PROB = 0.5         # 50% af billeder får støvpletter
SCRATCH_PROB = 0.3      # 30% får ridser
GRADIENT_PROB = 0.7     # 70% får ujævn belysning


def ellipses_overlap(e1, e2, gap):
    cx1, cy1, L1, _, _ = e1
    cx2, cy2, L2, _, _ = e2
    r1, r2 = L1 / 2.0, L2 / 2.0
    dx, dy = cx1 - cx2, cy1 - cy2
    return np.sqrt(dx*dx + dy*dy) < (r1 + r2 + gap)


def draw_ellipse(image, cx, cy, length, width, angle_deg, color):
    a, b = length / 2.0, width / 2.0
    rad = np.deg2rad(angle_deg)
    cos_a, sin_a = np.cos(rad), np.sin(rad)
    H, W = image.shape
    yy, xx = np.meshgrid(np.arange(H), np.arange(W), indexing='ij')
    xs, ys = xx - cx, yy - cy
    xr = cos_a * xs + sin_a * ys
    yr = -sin_a * xs + cos_a * ys
    image[(xr/a)**2 + (yr/b)**2 < 1.0] = color


def add_dust_spots(img, rng, max_spots=15):
    """Små mørke pletter — simulerer støv/skidt på pladen."""
    H, W = img.shape
    n_spots = int(rng.integers(3, max_spots + 1))
    yy, xx = np.ogrid[:H, :W]
    for _ in range(n_spots):
        cx = int(rng.integers(0, W))
        cy = int(rng.integers(0, H))
        r = int(rng.integers(1, 4))
        intensity = float(rng.uniform(0.30, 0.60))
        mask = (xx - cx) ** 2 + (yy - cy) ** 2 < r ** 2
        img[mask] = intensity


def add_scratches(img, rng, max_scratches=3):
    """Korte streger — simulerer ridser eller hår."""
    H, W = img.shape
    n = int(rng.integers(1, max_scratches + 1))
    for _ in range(n):
        x1 = int(rng.integers(0, W))
        y1 = int(rng.integers(0, H))
        length = int(rng.integers(15, 50))
        angle = float(rng.uniform(0, 2 * np.pi))
        x2 = int(x1 + length * np.cos(angle))
        y2 = int(y1 + length * np.sin(angle))
        intensity = float(rng.uniform(0.40, 0.70))
        n_points = max(abs(x2 - x1), abs(y2 - y1)) + 1
        xs = np.linspace(x1, x2, n_points).astype(int)
        ys = np.linspace(y1, y2, n_points).astype(int)
        valid = (xs >= 0) & (xs < W) & (ys >= 0) & (ys < H)
        img[ys[valid], xs[valid]] = intensity


def add_illumination_gradient(img, rng, max_strength=0.12):
    """Subtilt lineært lys-gradient — simulerer ikke-uniform belysning."""
    H, W = img.shape
    angle = float(rng.uniform(0, 2 * np.pi))
    strength = float(rng.uniform(0.04, max_strength))
    yy, xx = np.meshgrid(np.arange(H), np.arange(W), indexing='ij')
    gradient = xx * np.cos(angle) + yy * np.sin(angle)
    gradient = (gradient - gradient.min()) / (gradient.max() - gradient.min() + 1e-9)
    img += (gradient - 0.5) * strength


def generate_one_image(rng, num_seeds, img_size=256):
    """Ét billede med tilfældig støj + frø-størrelse + plade-imperfektioner."""
    noise_level = float(rng.choice(NOISE_LEVELS))
    L_min, L_max, W_min, W_max = SEED_SIZE_PRESETS[rng.integers(0, 3)]
    bg = float(rng.uniform(BACKGROUND_REFLECTANCE_MIN, BACKGROUND_REFLECTANCE_MAX))

    img = np.ones((img_size, img_size), dtype=np.float32) * bg
    placed, centers = [], []
    margin = L_max // 2 + 2

    for _ in range(num_seeds):
        L = int(rng.integers(L_min, L_max + 1))
        W = int(rng.integers(W_min, W_max + 1))
        for _ in range(MAX_PLACEMENT_ATTEMPTS):
            cx = int(rng.integers(margin, img_size - margin))
            cy = int(rng.integers(margin, img_size - margin))
            angle = float(rng.uniform(0, 360))
            new_e = (cx, cy, L, W, angle)
            if not any(ellipses_overlap(new_e, ex, PACKING_GAP_PIXELS) for ex in placed):
                color = float(rng.uniform(SEED_REFLECTANCE_MIN, SEED_REFLECTANCE_MAX))
                draw_ellipse(img, cx, cy, L, W, angle, color)
                placed.append(new_e)
                centers.append((cx, cy))
                break

    # v3: plade-imperfektioner FØR støj (de skal også støje)
    if rng.uniform() < DUST_PROB:
        add_dust_spots(img, rng)
    if rng.uniform() < SCRATCH_PROB:
        add_scratches(img, rng)
    if rng.uniform() < GRADIENT_PROB:
        add_illumination_gradient(img, rng)

    # Sensor-støj (sidste lag, så den ovenpå alt andet)
    noise = rng.standard_normal((img_size, img_size)) * noise_level
    img = np.clip(img + noise, 0.0, 1.0)

    return (img * 255).astype(np.uint8), centers, noise_level

print('Generator klar.')

## 3. Generér datasæt

1200 billeder, 80/20 split. **IMG_SIZE = 256** denne gang — match imgsz i træning
for at undgå unødvendig resize. Frø-størrelser i pixels giver mening på 256.

In [ ]:
NUM_IMAGES = 1200
IMG_SIZE = 256        # match imgsz i træning
BBOX_SIZE = 18
TRAIN_SPLIT = 0.8

dataset_dir = Path('dataset')
if dataset_dir.exists():
    shutil.rmtree(dataset_dir)
for split in ('train', 'val'):
    (dataset_dir / 'images' / split).mkdir(parents=True, exist_ok=True)
    (dataset_dir / 'labels' / split).mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(42)
indices = set(rng.permutation(NUM_IMAGES)[:int(NUM_IMAGES * TRAIN_SPLIT)].tolist())

noise_distribution = []
for i in range(NUM_IMAGES):
    n_seeds = int(rng.integers(40, 81))
    img, centers, noise = generate_one_image(rng, n_seeds, IMG_SIZE)
    noise_distribution.append(noise)
    split = 'train' if i in indices else 'val'

    Image.fromarray(img).save(dataset_dir / 'images' / split / f'img_{i:04d}.png')
    with open(dataset_dir / 'labels' / split / f'img_{i:04d}.txt', 'w') as f:
        for (cx, cy) in centers:
            x_n, y_n = cx / IMG_SIZE, cy / IMG_SIZE
            s_n = BBOX_SIZE / IMG_SIZE
            f.write(f'0 {x_n:.6f} {y_n:.6f} {s_n:.6f} {s_n:.6f}\n')

    if (i + 1) % 100 == 0:
        print(f'  {i+1}/{NUM_IMAGES} billeder')

(dataset_dir / 'data.yaml').write_text('train: images/train\nval: images/val\nnc: 1\nnames:\n  0: seed\n')

from collections import Counter
print(f'\nDatasæt klar: {len(indices)} train, {NUM_IMAGES - len(indices)} val')
print(f'Noise-distribution: {dict(Counter(round(n, 3) for n in noise_distribution))}')

## 3.5. Sanity-check: vis et augmenteret billede

Hurtig visuel verificering at dust/scratches/gradient faktisk er der.

In [ ]:
import matplotlib.pyplot as plt
import random

sample_files = list((dataset_dir / 'images' / 'train').glob('*.png'))
random.shuffle(sample_files)

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, f in zip(axes.flat, sample_files[:6]):
    ax.imshow(np.array(Image.open(f)), cmap='gray', vmin=0, vmax=255)
    ax.set_title(f.name, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()
print('Hvis du kan se variation i lysstyrke, små pletter, og evt. ridser — så virker det.')

## 4. Træning med aggressive augmentations

Oven i v2's flips og rotation tilføjer vi:
- `mixup=0.15` — blander to billeder lineært
- `copy_paste=0.3` — kopierer objekter mellem billeder
- `perspective=0.0005` — let kameravinkel-simulering
- `shear=5.0` — let skæv-deformation
- `patience=15` — mere tålmodig early stopping (træningen var ustabil i v2)

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11s.pt')

results = model.train(
    data='dataset/data.yaml',
    epochs=80,           # mere plads til konvergering
    imgsz=256,           # match dataset
    batch=32,
    patience=15,         # var 10 i v2
    project='runs',
    name='v3_augmented',
    exist_ok=True,
    device=0,
    verbose=True,
    # Spatial augmentations
    flipud=0.5,
    fliplr=0.5,
    degrees=180.0,
    perspective=0.0005,
    shear=5.0,
    # Mix-augmentations
    mixup=0.15,
    copy_paste=0.3,
)

print('\n=== TRÆNING FÆRDIG ===')
print(f'Precision: {results.box.mp:.4f}')
print(f'Recall:    {results.box.mr:.4f}')
print(f'mAP@50:    {results.box.map50:.4f}')
print(f'mAP@50-95: {results.box.map:.4f}')

## 5. Download best.pt

Næste skridt på din Mac:

```bash
mkdir -p models/v3_augmented
mv ~/Downloads/best.pt models/v3_augmented/
./switch_model.sh v3_augmented
fcm-demo
```

In [ ]:
import glob
from pathlib import Path
from google.colab import files

matches = glob.glob('**/best.pt', recursive=True)
if not matches:
    raise FileNotFoundError('Ingen best.pt fundet — fejlede træningen?')

best_pt = matches[0]
size_mb = Path(best_pt).stat().st_size / 1024 / 1024
print(f'Model: {best_pt}')
print(f'Størrelse: {size_mb:.2f} MB')
files.download(best_pt)